In [1]:
%matplotlib inline
import torch
import torchvision
from torch import nn
from d2l import torch as d2l
from torch.nn import functional as F

使用预训练的ResNet-18模型来提取图像特征

In [2]:
pretrained_net = torchvision.models.resnet18(pretrained=True)
list(pretrained_net.children())[-3:]

c:\Users\86180\miniconda3\envs\deepLearn\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\86180\miniconda3\envs\deepLearn\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[Sequential(
   (0): BasicBlock(
     (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (downsample): Sequential(
       (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
       (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     )
   )
   (1): BasicBlock(
     (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): Batc

创建一个全卷积网络实例net

In [3]:
net = nn.Sequential(*list(pretrained_net.children())[:-2])

X = torch.rand(size=(1, 3, 320, 480))
net(X).shape

torch.Size([1, 512, 10, 15])

构建FC

In [5]:
num_class = 21
net.add_module('final_conv', nn.Conv2d(512, num_class, kernel_size=1))

# 用双线性插值上采样替代转置卷积：无棋盘效应，不用算 k/p/s 的匹配关系
net.add_module('upsample', nn.Upsample(scale_factor=32, mode='bilinear', align_corners=True))


|参数	|含义|
|-------|----|
|scale_factor	|放大倍数，可以是单个数或 (H倍数, W倍数)|
|size	|目标 (H, W)，和 scale_factor 二选一|
|mode='bilinear'	|双线性插值|
|align_corners=True	|角点对齐，语义分割里一般设 True|